# SpaceX Falcon 9 First Stage Landing Prediction
## Module 3: Data Wrangling

**Author:** Pritam Acharya

The `Outcome` column from Module 1 records the raw landing result for each flight (e.g. `True ASDS`, `False Ocean`, `None None`) — a mix of whether the landing was attempted and whether it succeeded. This notebook converts that into a single binary **Class** label: `1` if the first stage landed successfully, `0` otherwise. This label is what the machine learning models in later modules will be trained to predict.


In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)


### Load the collected dataset

In [2]:
df = pd.read_csv("../data/dataset_part_1.csv")
df.head(5)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


### Identify missing values

In [3]:
(df.isnull().sum() / len(df) * 100).round(2)

FlightNumber       0.00
Date               0.00
BoosterVersion     0.00
PayloadMass        0.00
Orbit              0.00
LaunchSite         0.00
Outcome            0.00
Flights            0.00
GridFins           0.00
Reused             0.00
Legs               0.00
LandingPad        28.89
Block              0.00
ReusedCount        0.00
Serial             0.00
Longitude          0.00
Latitude           0.00
Class              0.00
dtype: float64

`LandingPad` is missing for ~29% of flights — that's expected: it's `NaN` whenever the booster attempted an ocean landing or made no landing attempt at all, since there's no physical pad involved.

### Column data types

In [4]:
df.dtypes

FlightNumber        int64
Date                  str
BoosterVersion        str
PayloadMass       float64
Orbit                 str
LaunchSite            str
Outcome               str
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad            str
Block             float64
ReusedCount         int64
Serial                str
Longitude         float64
Latitude          float64
Class               int64
dtype: object

### TASK 1: Launches by site
How many launches happened at each site?

In [5]:
df["LaunchSite"].value_counts()

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

### TASK 2: Launches by orbit type

In [6]:
df["Orbit"].value_counts()

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
ES-L1     1
HEO       1
SO        1
GEO       1
Name: count, dtype: int64

### TASK 3: Derive the landing outcome label

The raw `Outcome` string combines two pieces of information: whether the booster *attempted* a landing, and if so, whether it *succeeded*. We split outcomes into "bad" (landing failed, no attempt was made, or the outcome is unknown) versus everything else, and encode that as a binary `Class`.

In [7]:
df["Outcome"].value_counts()

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

In [8]:
bad_outcomes = set(outcome for i, outcome in enumerate(df["Outcome"].unique()) if outcome.split()[0] == "False" or outcome.split()[0] == "None")
bad_outcomes

{'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}

In [9]:
landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df["Outcome"]]
df["Class_derived"] = landing_class
df[["Outcome", "Class_derived"]].head(10)

,Outcome,Class_derived
0,None None,0
1,None None,0
2,None None,0
3,False Ocean,0
4,None None,0
5,None None,0
6,True Ocean,1
7,True Ocean,1
8,None None,0
9,None None,0


Sanity check: the label we just derived should match the `Class` column already present in the dataset (computed the same way upstream).

In [10]:
(df["Class"] == df["Class_derived"]).sum()

np.int64(90)

### Overall success rate

In [11]:
df["Class"].mean()

np.float64(0.6666666666666666)

About two-thirds (66.7%) of the 90 recorded landing attempts in this dataset succeeded — this is the baseline rate our later classification models need to beat.

### Export the wrangled dataset

In [12]:
df = df.drop(columns=["Class_derived"])
df.to_csv("../data/dataset_part_2.csv", index=False)
print(f"Saved {df.shape[0]} rows, {df.shape[1]} columns.")

Saved 90 rows, 18 columns.


### Summary

- Loaded the 90-row dataset collected via the SpaceX API (Module 1).
- Checked for missing values — only `PayloadMass` (already imputed) and `LandingPad` (meaningfully null) have gaps.
- Counted launches by site (4 sites) and orbit type (11 distinct orbits).
- Derived a binary landing-success `Class` label from the raw `Outcome` text, and verified it against the existing label — **90/90 rows match**.
- Overall first-stage landing success rate: **~60%**.
- Exported `dataset_part_2.csv` for exploratory analysis and modeling in the next modules.

**Next:** `4. jupyter-labs-eda-sql-coursera_sqllite.ipynb` — loading this data into a SQLite database and exploring it with SQL.
